# Week 8 Class Exercise: Prompt Engineering Lab: INSTRUCTOR SOLUTIONS

> ⚠️ **Instructor copy, do not distribute.** Solutions, model answers, grading notes, and the rubric are below. Exact scores depend on the model, so **run this notebook with your class provider before the session** to calibrate the expected numbers quoted in the grading notes.
### STAT / APPM 5350: Applied AI

**Team members:** _replace with both partners' names_

**Format:** pairs · **Time:** ~90 min in class, finish at home · **Submit:** this notebook (run, with outputs) + `prompt_log.csv`

In this lab you'll treat **prompting as an engineering discipline**: you will design prompts, change **one variable at a time**, and **measure** what changed, instead of judging by eye.

| Part | Topic (lecture slide) | Time |
|---|---|---|
| 0 | Setup and the experiment logger | 5 min |
| 1 | Prompt anatomy: role · task · constraints · output format | 10 min |
| 2 | Zero-shot vs few-shot (NER, job titles, multi-label) | 15 min |
| 3 | Model parameters: temperature and max tokens | 10 min |
| 4 | Hallucinations and guardrails | 10 min |
| 5 | Role-based prompting | 5 min |
| 6 | Chain-of-thought, answer extraction, selection-inference | 15 min |
| 7 | Prompt chaining and chunking | 10 min |
| 8 | Embeddings: similarity search and zero-shot classification | 10 min |
| 9 | **Capstone:** Lost and Found matching business case | at home |
| 10 | Prompt as an asset: reusable template and reflection | at home |

> **Ground rules.** Change *one* thing per experiment. Every call goes through `chat()`, which logs it automatically, so your submission documents your process. Where you see **✏️ TODO**, write code or prompts. Where you see **💭 Reflect**, write 2–4 sentences.

## 📋 Grading rubric (100 points)

| Part | Points | Full credit | Partial credit | No credit |
|---|---|---|---|---|
| **1** Prompt anatomy | **10** | v1→v3 each run and logged; v3 includes role, constraints, and an explicit JSON schema; pass rate reported (4). Robustness check run (2). Reflection names the most impactful change and a concrete fix for remaining failures (4). | v3 not reliably parseable, or versions skip a step (−2 to −4). Generic reflection (−2). | Prompts left as TODO. |
| **2** Zero vs few-shot | **12** | Few-shot prompt with 3–4 examples covering `none`, title-only output, and the pipe format, **with no test-set leakage** (5). Multi-label prompt constrained to the label set, mean Jaccard reported (4). Reflection addresses accuracy *vs* format consistency **and** the small-n limitation (3). | Examples miss a required case (−1 each). Labels not constrained (−2). | Reusing test sentences as examples: **max 5/12** for the part. |
| **3** Parameters | **8** | Temperature table and plot produced (2). max_tokens sweep run; truncation identified via `finish_reason` (2). Prompt-based length control finishes with `stop` (2). Reflection gives a sensible temperature per use case and explains truncation (2). | Length fixed only by raising max_tokens (−2). | — |
| **4** Hallucinations | **10** | Guarded prompt has context-only, exact abstention string, and evidence quote (5). Both prompts scored; guarded ≥ 4/5 (2). Reflection identifies a concrete hallucination and goes beyond prompting (retrieval, citations, HITL, monitoring) (3). | Guardrail lacks an exact abstention token, so scoring is unreliable (−2). | — |
| **5** Roles | **4** | Fourth role added and run (2). Reflection contrasts roles with specifics and proposes a combination strategy (2). | Vague comparison (−1). | — |
| **6** Reasoning | **12** | Two-stage (3) and selection-inference (4) implemented as **separate calls** that pass context forward. Accuracy *and* calls-per-question compared across all four (2). Reflection classifies errors as interpretation vs arithmetic and discusses cost (3). | Single-call imitation of a multi-call strategy (−2 each). | — |
| **7** Chaining & chunking | **8** | Chain runs end to end with JSON intermediates (2). Map-reduce implemented: map over chunks, then reduce with a topic line (4). Bonus question on boundaries/overlap answered (2). | Reduce step just concatenates (−2). | — |
| **8** Embeddings | **8** | Nearest-label classifier implemented with accuracy (4). Richer label descriptions tried and compared (2). Reflection weighs cost/latency/adaptability (and the TF-IDF limitation if relevant) (2). | — | — |
| **9** Capstone | **20** | Baseline reported (2). Normalization prompt with fixed schema, no invention, PII excluded; recall@k compared (5). Rerank prompt with hard-attribute rules and NO_MATCH (5). Results table complete with call counts (3). Memo covers workflow (HITL), scale/cost, **PII**, and index persistence (5). | ≥ 9/12 with 0 false matches earns full results credit; 7–8 → −2; false match on L12 → −2. | — |
| **10** Template & wrap-up | **8** | Template has role, inputs, schema, ≥ 2 real safety rules, eval numbers filled (4). `prompt_log.csv` exported (1). Final reflection answers all three questions with evidence from their own results (3). | Eval fields left null (−1). | — |

**Global deductions:** API key pasted into the notebook (**−5**, and tell the student to rotate it) · notebook submitted unrun or with errors (−5) · reflections that cite no numbers from their own runs (−1 each, max −5).

**Calibration tip:** in the grading notes below, "typical" results describe the *expected direction* of effects. Absolute numbers vary by model, so grade the **process and reasoning**, not whether a student hit a particular number.

## Part 0: Setup

Pick **one** provider and set the environment variables **before launching Jupyter** (or in a `.env` file). **Never paste API keys into the notebook.**

| `LLM_PROVIDER` | Required environment variables | Install |
|---|---|---|
| `openai` | `OPENAI_API_KEY`, optional `OPENAI_MODEL`, `OPENAI_EMBED_MODEL` | `pip install openai` |
| `azure` | `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT`, optional `AZURE_OPENAI_EMBED_DEPLOYMENT`, `AZURE_OPENAI_API_VERSION` | `pip install openai` |
| `watsonx` | `WATSONX_APIKEY`, `WATSONX_URL`, `WATSONX_PROJECT_ID`, optional `WATSONX_MODEL`, `WATSONX_EMBED_MODEL` | `pip install ibm-watsonx-ai` |
| `mock` | none (offline: returns placeholder text so you can read through the notebook) | — |

If no embedding model is configured, Part 8 falls back to TF-IDF vectors. That fallback is a useful baseline to compare against anyway.

In [ ]:
# %pip install -q openai ibm-watsonx-ai python-dotenv pandas numpy scikit-learn matplotlib

In [ ]:
import os, re, json, time, hashlib, itertools, textwrap
import numpy as np
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

PROVIDER = os.getenv("LLM_PROVIDER", "mock").lower()   # openai | azure | watsonx | mock
print("Provider:", PROVIDER)

_client = _wx_chat = _wx_embed = None
if PROVIDER == "openai":
    from openai import OpenAI
    _client = OpenAI()
    CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
elif PROVIDER == "azure":
    from openai import AzureOpenAI
    _client = AzureOpenAI(
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    )
    CHAT_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]           # deployment name
    EMBED_MODEL = os.getenv("AZURE_OPENAI_EMBED_DEPLOYMENT")     # may be None -> TF-IDF
elif PROVIDER == "watsonx":
    from ibm_watsonx_ai import Credentials
    from ibm_watsonx_ai.foundation_models import ModelInference, Embeddings
    _creds = Credentials(url=os.environ["WATSONX_URL"], api_key=os.environ["WATSONX_APIKEY"])
    CHAT_MODEL = os.getenv("WATSONX_MODEL", "ibm/granite-3-3-8b-instruct")
    EMBED_MODEL = os.getenv("WATSONX_EMBED_MODEL", "ibm/slate-125m-english-rtrvr")
    _wx_chat = ModelInference(model_id=CHAT_MODEL, credentials=_creds,
                              project_id=os.environ["WATSONX_PROJECT_ID"])
    _wx_embed = Embeddings(model_id=EMBED_MODEL, credentials=_creds,
                           project_id=os.environ["WATSONX_PROJECT_ID"])
else:
    CHAT_MODEL, EMBED_MODEL = "mock", None

print("Chat model:", CHAT_MODEL, "| Embedding model:", EMBED_MODEL or "TF-IDF fallback")

### The `chat()` helper and the experiment log

Every call is appended to `PROMPT_LOG` together with a `tag` (e.g. `"ex1-v2"`), the parameters, and the output. At the end you export it as your lab record. This mirrors how prompts are managed in practice: prompts are versioned assets, not throwaway text.

In [ ]:
PROMPT_LOG = []

def chat(prompt, system=None, temperature=0.0, max_tokens=400, tag="", show=True):
    """Send one prompt (plus an optional system message) and return the text response."""
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
    t0 = time.time()
    if PROVIDER in ("openai", "azure"):
        r = _client.chat.completions.create(model=CHAT_MODEL, messages=messages,
                                            temperature=temperature, max_tokens=max_tokens)
        out, finish = r.choices[0].message.content, r.choices[0].finish_reason
    elif PROVIDER == "watsonx":
        r = _wx_chat.chat(messages=messages,
                          params={"temperature": temperature, "max_tokens": max_tokens})
        out, finish = r["choices"][0]["message"]["content"], r["choices"][0].get("finish_reason")
    else:  # mock: deterministic placeholder (varies with temperature so Part 3 still runs)
        salt = str(np.random.rand()) if temperature > 0 else ""
        h = hashlib.md5((prompt + salt).encode()).hexdigest()[:6]
        out, finish = f"[mock response {h}]", "stop"
    PROMPT_LOG.append(dict(tag=tag, system=system, prompt=prompt, temperature=temperature,
                           max_tokens=max_tokens, output=out, finish_reason=finish,
                           latency_s=round(time.time() - t0, 2), model=CHAT_MODEL))
    if show:
        print(f"── [{tag or 'untagged'}] T={temperature} max_tokens={max_tokens} finish={finish}")
        print(textwrap.fill(out, 110, replace_whitespace=False) if out else out)
    return out


def parse_json(text):
    """Pull the first JSON object or array out of a model response. Returns None on failure."""
    if not text:
        return None
    m = re.search(r"(\{.*\}|\[.*\])", text, re.S)
    try:
        return json.loads(m.group(1)) if m else None
    except json.JSONDecodeError:
        return None

chat("In one sentence, what is a foundation model?", tag="smoke-test")

---
## Part 1: Prompt anatomy (Prompt Lab, Exercise 1)

A good prompt has four parts: **Role · Task · Constraints · Output format**. *Prompting is instruction design.*

Below is a weak prompt for a realistic task: triaging a customer email.

In [ ]:
EMAIL = """Hi, I ordered a standing desk (order #A-4471) three weeks ago. The tracking hasn't
updated in 10 days and your chat bot keeps closing my ticket. I need this for my home office
by the 20th or I'll have to cancel and buy elsewhere. Pretty frustrated. - Dana"""

weak = f"What about this email?\n\n{EMAIL}"
chat(weak, tag="ex1-v0-weak")

**✏️ TODO 1.1:** Improve the prompt in **three steps**, running each one so the log captures every version:
1. **v1** add a *role*
2. **v2** add *constraints* (e.g. don't invent order details; under 60 words for any free text)
3. **v3** enforce a *structured output*: JSON with keys `intent`, `urgency` (low/medium/high), `sentiment`, `order_id`, `deadline`, `suggested_reply`

Then check that `parse_json` returns a dict with exactly those keys.

In [ ]:
v1 = f"""You are a senior customer-support triage specialist for an online furniture retailer.
Analyze the customer email below and summarize what the customer needs.

{EMAIL}"""
chat(v1, tag="ex1-v1-role")

v2 = f"""You are a senior customer-support triage specialist for an online furniture retailer.
Analyze the customer email below: identify the customer's intent, urgency, sentiment, any order
number and deadline, and draft a short reply.

Constraints:
- Use only facts stated in the email; never invent tracking numbers, dates, or policies.
- If something is not mentioned, say "unknown".
- The reply must be under 60 words, empathetic, and must NOT promise a delivery date.

{EMAIL}"""
chat(v2, tag="ex1-v2-constraints")

v3 = f"""You are a senior customer-support triage specialist for an online furniture retailer.
Triage the customer email below.

Constraints:
- Use only facts stated in the email; never invent tracking numbers, dates, or policies.
- Use null for any field not mentioned in the email.
- suggested_reply: under 60 words, empathetic, no promised delivery dates.

Return ONLY a JSON object (no markdown fences, no commentary) with exactly these keys:
{{"intent": string, "urgency": "low" | "medium" | "high", "sentiment": string,
  "order_id": string | null, "deadline": string | null, "suggested_reply": string}}

<email>
{EMAIL}
</email>"""
out = chat(v3, tag="ex1-v3-format")

REQUIRED = {"intent", "urgency", "sentiment", "order_id", "deadline", "suggested_reply"}
parsed = parse_json(out)
print("\nValid JSON:", isinstance(parsed, dict),
      "| Missing keys:", REQUIRED - set(parsed or {}))

**✏️ TODO 1.2 (robustness check):** A format is only useful if it holds every time. Run `v3` **5 times at temperature 0.7** and report the fraction of responses that parse **and** contain all required keys.

In [ ]:
def schema_pass_rate(prompt, required, n=5, temperature=0.7, tag="ex1-robust"):
    ok = 0
    for _ in range(n):
        p = parse_json(chat(prompt, temperature=temperature, tag=tag, show=False))
        ok += isinstance(p, dict) and required <= set(p)
    return ok / n

print("Schema pass rate:", schema_pass_rate(v3, REQUIRED))

💭 **Reflect 1:** Which of the three additions changed the output most? What would you add to bring the pass rate to 100%? (Think about system messages, an example, or an explicit "Return only JSON".)

**Model answer:** The **output-format** step changes the most. v0/v1 produce free-form prose that a downstream system can't consume. v3 produces a parseable record with the exact keys, and the "use null / don't invent" constraint stops the model from fabricating a tracking number or a firm delivery date. The role mostly shifts tone and vocabulary. To push the pass rate to 100%: move the rules into a **system message**, add one **worked example** of the JSON, use the provider's **JSON / structured-output mode** where available, lower the temperature to 0 for extraction, and **validate and retry** on parse failure in code.

> 🧑‍🏫 **Grading notes:** Typical: v3 parses at 0.8–1.0 at T=0.7 with a strong model. Failures are usually Markdown ```json fences (which `parse_json` tolerates) or an extra key. Smaller open models fail more often, which makes a good discussion point. Give full credit only if the student names a *mechanism* (structured outputs, validate-and-retry, few-shot), not just "be clearer".

---
## Part 2: Zero-shot vs few-shot (Prompt Lab, Exercise 2)

Few-shot prompting shapes behavior **through context alone, without retraining**. We'll measure it on the **custom entity** task from the lecture: extracting **job titles**, where `none` is a valid answer.

In [ ]:
JOB_TEST = [
    ("Priya Natarajan joined Stripe last spring as a Staff Data Scientist.", "Staff Data Scientist"),
    ("Tom has lived in Denver for 12 years and loves skiing.", "none"),
    ("Contact: Luis Ortega | VP of Engineering | Plaid", "VP of Engineering"),
    ("After a decade as a nurse, Amara now leads the clinic as its Medical Director.", "Medical Director"),
    ("Kenji worked at Sony for 8 years and enjoyed every minute of it.", "none"),
    ("Our new Chief Revenue Officer, Beth Kim, starts Monday.", "Chief Revenue Officer"),
    ("Raj is the founder and CEO of a small robotics startup in Austin.", "CEO"),
    ("Elena teaches Spanish on weekends but is otherwise retired.", "Spanish teacher"),
]

def norm(s):
    return re.sub(r"[^a-z ]", "", (s or "").lower()).strip()

def evaluate(prompt_fn, data, tag, temperature=0.0):
    rows = []
    for x, y in data:
        pred = chat(prompt_fn(x), temperature=temperature, tag=tag, show=False)
        pred = pred.strip().split("\n")[0].replace("Job title:", "").strip()
        rows.append(dict(sentence=x, gold=y, pred=pred, correct=norm(pred) == norm(y)))
    df = pd.DataFrame(rows)
    print(f"{tag}: accuracy = {df.correct.mean():.0%}")
    return df

zero_shot = lambda s: f"Extract the job title from the following sentence.\n\nSentence: {s}\nJob title:"
df_zero = evaluate(zero_shot, JOB_TEST, tag="ex2-zero-shot")
df_zero

**✏️ TODO 2.1:** Write a **few-shot** prompt with 3–4 examples, using the `###` separator style from the lecture. Make sure your examples teach the model:
- to answer `none` when there is no job title,
- to return **only** the title (no name, no company),
- to handle the `Name | Title | Company` format.

⚠️ **Don't reuse test sentences as examples.** That is test-set leakage and inflates your accuracy.

In [ ]:
# Examples are the lecture's own job-title slide (NOT the test set), covering:
# a normal title, a "none" case, the pipe format, and a sentence with two roles/companies.
FEW_SHOT_HEADER = """Extract job titles from the following sentences.
Return only the job title exactly as written (no person or company name). If there is no job title, return none.

Sentence: Jane Doe has been working for Microsoft for 20 years as a Linux Engineer.
Job title: Linux Engineer
###
Sentence: Jane Doe has been working for Microsoft for 20 years and she loved it.
Job title: none
###
Sentence: Marc Simoncini | Director | Meetic
Job title: Director
###
Sentence: Damien is the CTO of Platform.sh, he was previously the CTO of Commerce Guys, a leading ecommerce provider.
Job title: CTO
###
"""
few_shot = lambda s: FEW_SHOT_HEADER + f"Sentence: {s}\nJob title:"
df_few = evaluate(few_shot, JOB_TEST, tag="ex2-few-shot")

pd.concat([df_zero.assign(setup="zero"), df_few.assign(setup="few")]) \
  .pivot_table(index="sentence", columns="setup", values="pred", aggfunc="first")

**✏️ TODO 2.2: Multi-label classification.** Using the movie-genre format from the lecture, build a few-shot prompt that returns a comma-separated list of genres from a **fixed label set** (`Action, Adventure, Comedy, Drama, Sport, Sci-Fi, Romance, Horror`). Score with **Jaccard similarity** between predicted and gold label sets, which is the right metric when partial credit matters.

In [ ]:
LABELS = ["Action", "Adventure", "Comedy", "Drama", "Sport", "Sci-Fi", "Romance", "Horror"]
MOVIES_TEST = [
    ("A washed-up boxer gets one last shot at the title while reconnecting with his estranged daughter.", {"Drama", "Sport"}),
    ("Astronauts stranded on a dying space station must outwit a rogue AI to get home.", {"Sci-Fi", "Action"}),
    ("Two rival wedding planners fall for each other while sabotaging the same event.", {"Comedy", "Romance"}),
    ("A family moves into a farmhouse where something in the cellar whispers their names at night.", {"Horror"}),
    ("A cartographer and a thief race across jungles and deserts to find a lost city before mercenaries do.", {"Action", "Adventure"}),
]

def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0

def movie_prompt(plot):
    return f"""Classify each movie plot into one to three genres chosen ONLY from this list:
{", ".join(LABELS)}.
Return only a comma-separated list of genres on one line, most relevant first.

Plot: As students at the United States Navy's elite fighter weapons school compete to be best in the class, one daring young pilot learns a few things from a civilian instructor that are not taught in the classroom.
Category: Drama, Action
###
Plot: A basketball scout discovers a phenomenal street ball player while in Spain and sees the prospect as his opportunity to get back into the NBA.
Category: Comedy, Drama, Sport
###
Plot: An action-filled epic that follows a young Viking prince on his quest to avenge his father's murder.
Category: Action, Adventure, Drama
###
Plot: {plot}
Category:"""

scores = []
for plot, gold in MOVIES_TEST:
    out = chat(movie_prompt(plot), tag="ex2-multilabel", show=False)
    pred = {l.strip() for l in out.split("\n")[0].split(",") if l.strip() in LABELS}
    scores.append(jaccard(pred, gold))
    print(f"{jaccard(pred, gold):.2f}  pred={sorted(pred)}  gold={sorted(gold)}")
print("Mean Jaccard:", round(np.mean(scores), 3))

💭 **Reflect 2:** Did few-shot improve **accuracy**, **consistency of format**, or both? Which test item was hardest and why? (Hint: look at "Spanish teacher" and "founder and CEO".) Is 8 test items enough to conclude anything? What would you do in a real evaluation?

**Model answer:** Few-shot mainly improves **format consistency**: zero-shot answers like "The job title is Staff Data Scientist at Stripe" or "There is no job title mentioned" fail exact-match scoring even when the model "knows" the answer. The examples teach the `none` convention and title-only output. The hardest items are **ambiguous by design**. "Elena teaches Spanish" implies a role without stating a title, and the retirement clause means `none` is defensible. "Founder and CEO" has two titles, so the gold label is itself a judgment call. With **8 items**, one flip moves accuracy by 12.5 points, so no claim is statistically meaningful. A real evaluation needs a few hundred labeled items, a held-out split, inter-annotator agreement on the gold labels, and confidence intervals (e.g. bootstrap), plus multiple runs if T > 0.

> 🧑‍🏫 **Grading notes:** Typical: zero-shot 40–75% (mostly formatting misses), few-shot 75–100%. **Check for leakage**: any example sentence copied from `JOB_TEST` caps the part at 5/12. Reward students who question the gold labels themselves. That is exactly the right instinct. For multi-label, mean Jaccard is typically 0.6–0.9, and the misses are usually an extra genre (e.g. "Drama" added to everything, a bias picked up from the examples).

---
## Part 3: Model parameters (Prompt Lab, Exercise 3)

**Temperature** controls sampling randomness. **Max tokens** caps output length and truncates the output, it does *not* make the model more concise. We'll quantify both.

In [ ]:
def diversity(outputs):
    """Fraction of unique outputs, and mean pairwise word-level Jaccard distance."""
    sets = [set(o.lower().split()) for o in outputs]
    dists = [1 - jaccard(a, b) for a, b in itertools.combinations(sets, 2)]
    return len(set(outputs)) / len(outputs), float(np.mean(dists)) if dists else 0.0

TAGLINE = "Write a tagline for an ice cream shop. Return only the tagline."
rows = []
for T in [0.0, 0.5, 1.0, 1.5]:
    outs = [chat(TAGLINE, temperature=T, tag=f"ex3-temp-{T}", show=False) for _ in range(5)]
    uniq, dist = diversity(outs)
    rows.append(dict(temperature=T, unique_frac=uniq, mean_jaccard_dist=round(dist, 3), sample=outs[0]))
df_temp = pd.DataFrame(rows)
df_temp

In [ ]:
import matplotlib.pyplot as plt
ax = df_temp.plot(x="temperature", y=["unique_frac", "mean_jaccard_dist"], marker="o", figsize=(6, 3.5))
ax.set_ylabel("diversity"); ax.set_title("Output diversity vs temperature (n=5 per setting)"); plt.show()

**✏️ TODO 3.1:** Run the summarization prompt below with `max_tokens` = 20, 60, 200. Look at `finish_reason` (`length` means truncated). Then fix the 20-token case **through the prompt instead of the parameter**: ask for a ≤15-word summary and check that it now finishes with `stop`.

In [ ]:
ARTICLE = """Global warming refers to the gradual increase in the overall temperature of the Earth's
atmosphere, primarily caused by the burning of fossil fuels such as coal, oil, and natural gas.
This burning releases carbon dioxide and other greenhouse gases into the atmosphere, which trap
heat and cause the Earth's temperature to rise. Governments around the world need to take
immediate action to reduce carbon emissions and invest in renewable energy sources."""

for mt in [20, 60, 200]:
    chat(f"Summarize:\n\n{ARTICLE}", max_tokens=mt, tag=f"ex3-maxtok-{mt}")

# Length control via the prompt: same 20-token budget, but the model now plans a short answer
chat(f"Summarize the text below in ONE sentence of at most 15 words. Return only that sentence.\n\n{ARTICLE}",
     max_tokens=20, tag="ex3-maxtok-20-prompted")

💭 **Reflect 3:** Which temperature would you use for (a) extracting fields into JSON, (b) brainstorming marketing taglines, (c) the capstone matching task? Why does `max_tokens` make a poor length control?

**Model answer:** (a) **JSON extraction: T = 0**. We want the single most likely, reproducible answer, and randomness only adds schema failures. (b) **Taglines: T ≈ 0.8–1.2**. Diversity is the goal, and a human picks the best (the lecture's "generate many outputs, then pick the best"). (c) **Capstone matching: T = 0**. It is a decision task where reproducibility and auditability matter. `max_tokens` is a poor length control because it is a **hard cutoff applied after generation starts**: the model doesn't know about the limit, plans a full-length answer, and gets cut mid-sentence (`finish_reason = "length"`). Asking for a length in the prompt changes what the model *plans* to write. Keep `max_tokens` as a cost/safety ceiling.

> 🧑‍🏫 **Grading notes:** Typical: unique_frac ≈ 0.2 at T=0 (often 1 unique output out of 5) rising to 1.0 by T≥1. At T=1.5 students may see degraded or odd taglines, which is worth showing in class. Note that T=0 is **not guaranteed deterministic** on hosted APIs (batching and floating-point effects). Bonus discussion if a student observes this. The 20-token run should show `finish=length`, and the prompted version `finish=stop`.

---
## Part 4: Hallucinations and guardrails (Prompt Lab, Exercise 4)

*AI should signal uncertainty, and good prompts limit speculation.* Below is the Microsoft/Azure OpenAI news passage from the lecture's Q&A slide. Some questions **can** be answered from it and some **cannot**.

In [ ]:
PASSAGE = """On Monday, Microsoft launched an OpenAI service as part of its Azure cloud platform,
offering businesses and start-ups the ability to incorporate models like ChatGPT into their own
systems. The company has already been building AI tools into many of its consumer products, such
as a DALL-E 2 feature in its Bing search engine that can create images based on a text prompt, and
the Information reported recently that it's working to bring more of them to Microsoft Office as well."""

QA = [  # (question, answerable_from_passage?)
    ("What did Microsoft launch on Monday?", True),
    ("Which search engine has a DALL-E 2 feature?", True),
    ("How much does the Azure OpenAI service cost per 1,000 tokens?", False),
    ("Who is the CEO who announced the launch?", False),
    ("Which Microsoft Office apps will get AI tools first?", False),
]

naive = lambda q: f"{PASSAGE}\n\nQuestion: {q}\nAnswer:"
for q, _ in QA:
    chat(naive(q), tag="ex4-naive")

**✏️ TODO 4.1:** Write a **guarded** prompt. It must (1) answer **only** from the passage, (2) reply with exactly `NOT IN CONTEXT` when the passage doesn't contain the answer, and (3) quote the supporting sentence when it does answer. Then score both prompts on **abstention accuracy**: the model should abstain on the unanswerable questions and answer the rest.

In [ ]:
def guarded(q):
    return f"""You are a careful research assistant. Answer the question using ONLY the passage below.

Rules:
1. If the passage does not explicitly contain the answer, reply with exactly: NOT IN CONTEXT
2. Do not use outside knowledge, even if you believe you know the answer.
3. If you do answer, add a second line: Evidence: "<exact quote from the passage>"

Passage:
<passage>
{PASSAGE}
</passage>

Question: {q}
Answer:"""

def abstention_score(prompt_fn, tag):
    correct = 0
    for q, answerable in QA:
        out = chat(prompt_fn(q), tag=tag, show=False)
        abstained = "NOT IN CONTEXT" in out.upper()
        correct += (abstained != answerable)
        print(f"{'✓' if abstained != answerable else '✗'} abstained={abstained!s:5}  {q}")
    print(f"{tag}: {correct}/{len(QA)}\n")

abstention_score(naive, "ex4-naive")
abstention_score(guarded, "ex4-guarded")

💭 **Reflect 4:** Did the naive prompt hallucinate (e.g. invent a price or a CEO name)? Is a guardrail prompt *sufficient* for a production system? What else would you add (retrieval, citations, human review)? Connect your answer to the **Responsible AI takeaway** slide.

**Model answer:** The naive prompt scores at most 2/5 by construction, since it never emits the abstention token. The more important finding is *what* it says on the unanswerable questions. Strong models often answer "Who is the CEO?" with **Satya Nadella** from parametric memory. That is plausibly true but **unsupported by the passage**, which is a grounding failure even when the fact is correct. On pricing and Office apps, models either hedge or invent specifics. The guarded prompt reaches 5/5 here, but prompting alone isn't sufficient for production: the model can still over-abstain or paraphrase a "quote". A production system needs retrieval over a trusted corpus (RAG), **citation verification in code** (check that the quoted evidence actually appears in the source), confidence thresholds with human review for low-confidence or high-stakes answers, logging, and ongoing evaluation. This is the Responsible AI point: *AI should signal uncertainty, and good prompts limit speculation, but systems enforce it.*

> 🧑‍🏫 **Grading notes:** Give full credit only if the student names a concrete hallucination from *their* run and at least one non-prompt mitigation. Nice extension to suggest: `assert evidence in PASSAGE`.

---
## Part 5: Role-based prompting (Prompt Lab, Exercise 5)

Same task, different roles. Compare **tone, emphasis, and what gets left out**.

In [ ]:
TASK = ("Explain to a theme-park operations team why we should use a large language model to "
        "help match lost-item reports with found items. 4 sentences max.")

ROLES = {
    "data_scientist": "You are a senior data scientist who cares about measurable accuracy.",
    "privacy_officer": "You are a privacy and compliance officer focused on PII and risk.",
    "ops_manager": "You are a busy operations manager who cares about staff time and guest satisfaction.",
    "guest_experience": "You are a guest-experience lead who thinks about the emotional moment of losing "
                        "something precious on vacation, and about the park's reputation.",
}
role_outputs = {r: chat(TASK, system=s, temperature=0.3, tag=f"ex5-{r}") for r, s in ROLES.items()}

💭 **Reflect 5:** List one point each role raised that the others didn't. If you were writing a product spec, how would you combine them? (One option is a chain: draft as ops manager, then critique as privacy officer.)

**Model answer:** The *data scientist* emphasizes evaluation (precision/recall, a pilot against historical matches). The *privacy officer* raises PII in reports (names, IDs, photos), data retention, and vendor terms, and usually hedges on adoption. The *ops manager* talks about staff hours, queue times, and a simple workflow. The *guest-experience lead* focuses on speed to reunite guests with sentimental items and on proactive notification. No single role covers the full picture, and each **omits** the others' concerns. That omission is the real risk of role prompting. For a spec, use a **chain**: draft as ops manager (value and workflow), critique as privacy officer (risks and controls), then have the data scientist add success metrics and a pilot plan, and finally synthesize. This is the lecture's "ask for justifications of many possible answers, then synthesize".

> 🧑‍🏫 **Grading notes:** Look for specific contrasts quoted from their outputs, not generic "the tone was different".

---
## Part 6: Reasoning: chain-of-thought, answer extraction, selection-inference

The lecture showed that asking the model to **reason before answering** fixes errors like the *8,000,000 liters* water-demand answer. We'll test this on a small benchmark with **known answers** and compare four strategies.

In [ ]:
MATH = [
    ("What is the annual water demand of a single-family household containing four people who are "
     "at home an average of 200 days per year and use an average of 100 liters of water per day "
     "(per person)?", 80000),
    ("On average Joe throws 25 punches per minute. A fight lasts 5 rounds of 3 minutes. "
     "How many punches did he throw?", 375),
    ("It takes Amy 4 minutes to climb to the top of a slide and 1 minute to slide down. "
     "The water slides close in 15 minutes. How many times can she slide before it closes?", 3),
    ("The cafeteria has 23 apples. If they used 20 to make lunch and bought 6 more, "
     "how many do they have?", 9),
    ("A store sells pens in packs of 12. A school needs 150 pens. How many packs must it buy?", 13),
    ("A train leaves at 9:40 and arrives at 13:05. How many minutes is the trip?", 205),
]

def last_number(text):
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text or "")
    return float(nums[-1].replace(",", "")) if nums else None

def run_strategy(name, solve):
    rows = []
    for q, gold in MATH:
        pred = solve(q)
        rows.append(dict(strategy=name, question=q[:55] + "…", gold=gold, pred=pred, correct=pred == gold))
    return pd.DataFrame(rows)

# Strategy A: direct answer
def direct(q):
    return last_number(chat(f"{q}\nAnswer with a number only.", tag="ex6-direct", show=False))

# Strategy B: zero-shot chain-of-thought ("Let's think step by step")
def zero_cot(q):
    return last_number(chat(f"{q}\nLet's think step by step, then give the final answer "
                            f"on the last line as 'Answer: <number>'.", tag="ex6-cot", show=False))

results = pd.concat([run_strategy("A_direct", direct), run_strategy("B_zero_shot_CoT", zero_cot)])
results.groupby("strategy").correct.mean()

**✏️ TODO 6.1:** Implement the two remaining strategies from the lecture:

- **C: Two-stage answer extraction.** Call 1 produces the reasoning. Call 2 receives the question **plus that reasoning** and is asked only for the final number (*"Therefore, the answer (Arabic numerals) is"*).
- **D: Selection-inference.** Call 1 asks the model to *decompose* the problem into the sub-question it must solve first. Call 2 answers that sub-question. Call 3 adds the new fact to the context and answers the original question.

Add both to `results` and compare all four. Track the **number of calls / tokens** too, since reasoning has a cost.

In [ ]:
def two_stage(q):
    # Call 1: reason extraction
    reasoning = chat(f"{q}\nLet's think step by step and explain the calculation step by step.",
                     tag="ex6-2stage-reason", show=False)
    # Call 2: answer extraction, conditioned on the question + the model's own reasoning
    final = chat(f"{q}\n{reasoning}\nTherefore, the answer (Arabic numerals only) is",
                 max_tokens=20, tag="ex6-2stage-extract", show=False)
    return last_number(final)

def selection_inference(q):
    # Call 1: decompose into the sub-question to solve first
    sub_q = chat(f"{q}\nTo solve this, what is the ONE sub-question we must answer first? "
                 "Return only the sub-question.", max_tokens=60, tag="ex6-si-decompose", show=False)
    # Call 2: answer the sub-question
    fact = chat(f"{q}\nSub-question: {sub_q}\nAnswer the sub-question in one sentence.",
                max_tokens=80, tag="ex6-si-subanswer", show=False)
    # Call 3: add the new fact to the context and answer the original question
    final = chat(f"{q}\nKnown fact: {fact}\nUsing this fact, solve the problem and give the final "
                 "answer on the last line as 'Answer: <number>'.", tag="ex6-si-final", show=False)
    return last_number(final)

results = pd.concat([results,
                     run_strategy("C_two_stage", two_stage),
                     run_strategy("D_selection_inference", selection_inference)])

CALLS_PER_Q = {"A_direct": 1, "B_zero_shot_CoT": 1, "C_two_stage": 2, "D_selection_inference": 3}
summary = results.groupby("strategy").correct.mean().to_frame("accuracy")
summary["calls_per_question"] = summary.index.map(CALLS_PER_Q)
summary

💭 **Reflect 6:** Which items did the direct strategy get wrong, and was the error arithmetic or *problem interpretation* (e.g. per-person vs per-household, rounding up packs)? When is the extra cost of CoT or multi-call prompting worth it in a business setting?

**Model answer:** Direct-answer errors are mostly **interpretation**, not arithmetic: the *packs of pens* question (150/12 = 12.5 needs rounding **up** to 13), the *train* question (time arithmetic across hours), and *Amy's slide* (the 4+1-minute cycle). Note that the lecture slide's own "after" answer to the water question (**20,000 L**) is also wrong, because it drops the 4 people. That is a CoT answer that *looks* rigorous but misreads the problem, which is why our version says "(per person)" and the gold is 4 × 100 × 200 = **80,000 L**. CoT helps by making the model commit to intermediate quantities, and multi-call variants let us inspect or verify each step. The cost is 2–3× the calls and many more output tokens, so latency and spend grow. It's worth it for **high-stakes or multi-step quantitative decisions** (pricing, dosage, financial calculations) and when auditability matters. It's not worth it for simple classification or extraction, or at high volume where a direct prompt is already accurate. Better still, hand arithmetic to code (tool use).

> 🧑‍🏫 **Grading notes:** Typical with a modern chat model: direct 50–85%, CoT strategies 85–100%. Modern models often reason implicitly, so the gap is smaller than the 2022 slides suggest. Discuss why. Penalize C/D that are implemented as a *single* call. **Point out the slide's 20,000 L error in class**, since it makes a great "trust but verify" moment.

---
## Part 7: Prompt chaining and chunking

**Chaining:** the output of one prompt becomes the input to the next (Entity extraction → Summarization → Sentiment). Using **structured intermediate outputs** (JSON) makes chains reliable.

In [ ]:
NEWS = ("The new iPhone model is set to be released next month. It has been highly anticipated by "
        "Apple fans and is expected to feature a larger screen and improved camera, though analysts "
        "warn the price may rise by as much as $100.")

step1 = chat(f"""Extract named entities from the article. Return only a JSON list of objects
with keys "text" and "type" (PRODUCT, ORGANIZATION, PERSON, MONEY, DATE).

Article: {NEWS}""", tag="ex7-chain-1-entities")
entities = parse_json(step1) or []

step2 = chat(f"""Using ONLY the article below, write a 2-sentence summary focused on these entities:
{json.dumps(entities)}

Article: {NEWS}""", tag="ex7-chain-2-summary")

step3 = chat(f"""Classify the sentiment of the text as positive, neutral, negative, or mixed.
Return JSON: {{"sentiment": ..., "evidence": "<short quote>"}}

Text: {step2}""", tag="ex7-chain-3-sentiment")

⚠️ Notice that the lecture's version of step 2 summarized *the product in general* ("The iPhone is a line of smartphones…") because it passed **only the entities** and not the article. That is a classic chaining bug where context is lost between steps. Our version passes both. Does the price warning change the sentiment from the lecture's "Positive"?

**✏️ TODO 7.1: Chunking (map-reduce).** Models have context limits, and long inputs often degrade quality. Implement `map_reduce_summary(text, chunk_words)`:
1. **Map:** split the text into chunks of about `chunk_words` words and summarize each one (1–2 sentences).
2. **Reduce:** summarize the chunk summaries into a final summary plus a `Topic:` line.

Test it on `LONG_TEXT`. Bonus: split on sentence boundaries with overlap and explain why that helps.

In [ ]:
LONG_TEXT = (ARTICLE + " ") * 2 + """Climate change is leading to more frequent and severe natural
disasters, including floods, wildfires, and droughts. Coastal cities face rising sea levels that
threaten infrastructure and housing. Farmers are adapting by changing crop varieties and planting
seasons. Meanwhile, renewable energy costs have fallen sharply, with solar and wind now cheaper than
new coal plants in most markets. Failure to act will have dire consequences for future generations."""

def chunk(text, chunk_words=60):
    words = text.split()
    return [" ".join(words[i:i + chunk_words]) for i in range(0, len(words), chunk_words)]

def chunk_sentences(text, max_words=60, overlap=1):
    """Pack whole sentences into chunks of <= max_words, carrying `overlap` sentences forward."""
    sents = re.split(r"(?<=[.!?])\s+", " ".join(text.split()))
    chunks, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur + [s]).split()) > max_words:
            chunks.append(" ".join(cur))
            cur = cur[-overlap:] if overlap else []
        cur.append(s)
    if cur:
        chunks.append(" ".join(cur))
    return chunks

def map_reduce_summary(text, chunk_words=60):
    parts = chunk_sentences(text, chunk_words)
    # Map: summarize each chunk independently
    partials = [chat(f"Summarize this excerpt in 1-2 sentences using only its content.\n\nExcerpt: {c}",
                     max_tokens=120, tag="ex7-map", show=False) for c in parts]
    bullets = "\n".join(f"- {p}" for p in partials)
    # Reduce: merge the partial summaries, removing duplicates
    return chat(f"""Below are summaries of consecutive sections of ONE document.
Write a 3-sentence overall summary that does not repeat points, then a final line 'Topic: <3-6 words>'.

{bullets}""", max_tokens=200, tag="ex7-reduce")

print(len(chunk(LONG_TEXT)), "word chunks |", len(chunk_sentences(LONG_TEXT)), "sentence chunks")
map_reduce_summary(LONG_TEXT)

> 🧑‍🏫 **Grading notes (Part 7):** **Bonus answer:** fixed word windows cut sentences in half (e.g. "…such as coal, oil" | "and natural gas…"), so a chunk summary can lose or distort a claim. Splitting on sentence boundaries keeps claims intact, and **overlap** preserves context (pronouns, "this burning…") that crosses a boundary. The cost is some duplicated tokens. `LONG_TEXT` deliberately repeats the article twice, so a good reduce prompt must **deduplicate**. Students who notice that repeated content in the source shows up as repeated summary points deserve credit. On the chain: with the price-warning sentence added, sentiment should come out **mixed** rather than the lecture's "Positive", which shows how a lossy step 2 changes downstream conclusions.

---
## Part 8: Embeddings: similarity search and zero-shot classification

Embeddings map text to vectors where **semantic similarity becomes cosine similarity**. Uses from the lecture: search, clustering, recommendations, anomaly detection, and classification. This is also the foundation of RAG, which comes next in the course: *Prompting → RAG → Agents → Deployment*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def embed(texts, fit_corpus=None):
    """Return an (n, d) array of embeddings. Falls back to TF-IDF if no embedding model is set."""
    if PROVIDER in ("openai", "azure") and EMBED_MODEL:
        r = _client.embeddings.create(model=EMBED_MODEL, input=texts)
        return np.array([d.embedding for d in r.data])
    if PROVIDER == "watsonx":
        return np.array(_wx_embed.embed_documents(texts=texts))
    vec = TfidfVectorizer(stop_words="english").fit(fit_corpus or texts)
    return vec.transform(texts).toarray()

EMBED_BACKEND = "TF-IDF" if (PROVIDER == "mock" or (PROVIDER in ("openai", "azure") and not EMBED_MODEL)) else EMBED_MODEL
print("Embedding backend:", EMBED_BACKEND)

PRODUCTS = [
    "Mid-century walnut dining table that seats six",
    "Ergonomic mesh office chair with lumbar support",
    "Three-seat velvet sofa in emerald green",
    "Adjustable-height standing desk with bamboo top",
    "Outdoor teak lounge chair, weather resistant",
    "Kids' bunk bed with built-in storage drawers",
    "Glass coffee table with brass legs",
    "Memory foam queen mattress, medium firm",
]
QUERIES = ["somewhere comfy for my back while I work from home",
           "a couch for the living room",
           "furniture for the patio"]

P = embed(PRODUCTS, fit_corpus=PRODUCTS + QUERIES)
Q = embed(QUERIES, fit_corpus=PRODUCTS + QUERIES)
for q, sims in zip(QUERIES, cosine_similarity(Q, P)):
    top = np.argsort(-sims)[:3]
    print(f"\n🔎 {q}")
    for i in top:
        print(f"   {sims[i]:.3f}  {PRODUCTS[i]}")

> 🔍 **Look closely:** on the TF-IDF fallback every score above is **0.000**, because the queries share no words with the products ("couch" vs "sofa"). With a real embedding model, "couch" lands next to "sofa" and "patio" next to "outdoor teak". That gap between lexical and semantic matching is the whole reason embeddings exist. If you're on a real provider, temporarily set `EMBED_MODEL = None` to see the contrast.

**✏️ TODO 8.1: Zero-shot classification with embeddings.** Following the OpenAI Cookbook approach on the slide, classify reviews by embedding **label descriptions** (e.g. `"A positive product review"`) and assigning each review to its nearest label. Report accuracy. Then try *richer* label descriptions and see if accuracy moves.

If you are on the TF-IDF fallback, explain why it struggles here. (Hint: lexical overlap is not the same as meaning.)

In [ ]:
REVIEWS = [
    ("Arrived broken and support never answered my emails.", "negative"),
    ("Absolutely love it, my kids use it every day!", "positive"),
    ("It's fine. Does what it says, nothing special.", "neutral"),
    ("Stopped working after two weeks. Waste of money.", "negative"),
    ("Exceeded expectations, would buy again in a heartbeat.", "positive"),
    ("Average quality for the price; shipping was on time.", "neutral"),
]
LABEL_DESCRIPTIONS = {
    "positive": "A positive product review",
    "negative": "A negative product review",
    "neutral":  "A neutral product review",
}

def zero_shot_embed(label_desc, name):
    texts, labels = [r for r, _ in REVIEWS], list(label_desc)
    corpus = texts + list(label_desc.values())
    R = embed(texts, fit_corpus=corpus)
    L = embed(list(label_desc.values()), fit_corpus=corpus)
    preds = [labels[i] for i in cosine_similarity(R, L).argmax(axis=1)]
    acc = np.mean([p == g for p, (_, g) in zip(preds, REVIEWS)])
    print(f"{name}: accuracy = {acc:.0%}   preds = {preds}")
    return acc

zero_shot_embed(LABEL_DESCRIPTIONS, "short labels")

RICH_LABELS = {
    "positive": "A delighted customer praising the product: love it, great quality, exceeded expectations, would buy again.",
    "negative": "An unhappy customer complaining: arrived broken, stopped working, bad support, waste of money, refund.",
    "neutral":  "A lukewarm customer review: it's okay, fine, average, nothing special, does the job.",
}
zero_shot_embed(RICH_LABELS, "rich labels")

💭 **Reflect 8:** When would you choose embeddings plus nearest-neighbor over prompting an LLM to classify? Consider cost, latency, explainability, and how easily each one adapts to new labels.

**Model answer:** Short labels like "A positive product review" differ by one word, so the label vectors are nearly identical and accuracy is often near chance. Richer descriptions spread the label vectors apart and typically help a lot. On **TF-IDF**, accuracy depends entirely on word overlap with the label text ("broken", "love"), so the short labels fail and the rich labels "work" only because we pasted in keywords from the test set. That is a form of leakage worth naming. Choose **embeddings + nearest neighbor** when volume is high and cost/latency matter (one cheap embedding per item, which can be cached and batched), when you have labeled examples to use as centroids, or when you need similarity for search or dedup anyway. Choose **LLM prompting** when labels need reasoning or nuance (sarcasm, mixed sentiment), when you need an explanation, or when labels change often and you have no examples. Adding a label is a prompt edit, while for embeddings it is a new description to tune. A common production pattern combines them: embeddings to shortlist, an LLM to decide (exactly the capstone design).

> 🧑‍🏫 **Grading notes:** Typical with real embeddings: short labels 33–67%, rich labels 83–100%. With TF-IDF, short labels are often ~33% (frequently all one class). Reward students who spot the keyword-leakage issue with rich labels.

---
## Part 9: 🏰 Capstone: Lost and Found matching (business case)

> *The Lost and Found Matching department at a large theme park wants to use GenAI to cut the time spent matching found items to lost reports by **50%** after the July 4th rush.*

Your mission is to **match lost reports to found items** using the techniques from today, and to **measure** how well it works. The data below is synthetic. Found-item logs are written tersely by staff, while lost reports are written by guests in their own words, so exact keyword matching won't work.

**Pipeline to build:**
1. **Normalize** (prompting + JSON): extract structured attributes (`category`, `color`, `brand`, `distinguishing_features`, `location`) from every report.
2. **Retrieve** (embeddings): for each lost report, shortlist the top-k found items by cosine similarity.
3. **Rerank** (LLM + reasoning): ask the model to pick the best candidate among the top-k **or answer `NO_MATCH`**, and justify the choice.
4. **Evaluate:** top-1 accuracy, recall@k for retrieval, and how often it says NO_MATCH correctly. Lost report `L12` has **no** matching found item.

**Target:** ≥ 10 correct matches out of 11 matchable reports, with zero false matches for `L12`.

In [ ]:
LOST = pd.DataFrame([
    ("L01", "07-04", "Main Street",     "Lost my daughter's pink Minnie ears headband with a sequin bow, her name 'Ava' written inside."),
    ("L02", "07-04", "Tomorrowland",    "Black iPhone 15 in a clear case with a Polaroid photo of a golden retriever tucked behind it."),
    ("L03", "07-04", "Adventureland",   "Navy blue Patagonia backpack, has a broken left zipper pull and a carabiner with keys."),
    ("L04", "07-04", "Fantasyland",     "My son's stuffed Stitch toy, really worn out, missing one ear tip, he can't sleep without it!"),
    ("L05", "07-05", "Frontierland",    "Prescription sunglasses, tortoiseshell Ray-Ban frames, in a brown leather case."),
    ("L06", "07-05", "Main Street",     "Brown leather wallet, Coach brand, has a Colorado driver's license and a Costco card."),
    ("L07", "07-05", "Liberty Square",  "Gray baby stroller blanket with yellow stars, hand knit by grandma."),
    ("L08", "07-05", "Tomorrowland",    "Red refillable water bottle (Hydro Flask) covered in national park stickers."),
    ("L09", "07-04", "Fireworks viewing near castle", "Gold wedding band, plain, engraved 'J & M 2019' inside. Desperate to find it."),
    ("L10", "07-05", "Adventureland",   "Kids' light-up sneakers, size 11 toddler, Paw Patrol, left shoe only."),
    ("L11", "07-04", "Fantasyland",     "Green Nintendo Switch Lite in a Zelda carrying case, game cartridge inside."),
    ("L12", "07-05", "Frontierland",    "Silver Apple Watch with an orange sport band."),
], columns=["lost_id", "date", "location", "description"])

FOUND = pd.DataFrame([
    ("F01", "07-04", "Fantasyland - Carousel",    "Plush toy, blue alien character, heavily worn, ear damaged."),
    ("F02", "07-05", "Frontierland - Big Thunder","Eyeglasses w/ case. Brown case, patterned frames, Rx lenses."),
    ("F03", "07-04", "Main St - Emporium",        "Headband, Minnie ears, pink w/ bow. Writing on inside band."),
    ("F04", "07-05", "Main St - Casey's",         "Wallet, brown leather. ID inside (CO). Membership cards."),
    ("F05", "07-04", "Tomorrowland - Space Mtn",  "Cell phone, black, clear case. Photo of dog in case. Locked."),
    ("F06", "07-05", "Tomorrowland - Speedway",   "Water bottle, red, insulated, many stickers."),
    ("F07", "07-04", "Hub grass area",            "Ring, gold band, inscription inside."),
    ("F08", "07-05", "Adventureland - Jungle Cruise", "Single child's shoe, light-up, cartoon dogs, small."),
    ("F09", "07-04", "Adventureland - Pirates",   "Backpack, dark blue, keys attached on clip. Zipper damaged."),
    ("F10", "07-05", "Liberty Sq - Haunted Mansion", "Blanket, small, knitted, gray with star pattern."),
    ("F11", "07-04", "Fantasyland - Peter Pan",   "Handheld game console, green, in case w/ game."),
    # distractors
    ("F12", "07-04", "Tomorrowland - Space Mtn",  "Cell phone, black, no case, cracked screen."),
    ("F13", "07-05", "Main St - City Hall",       "Wallet, black nylon, velcro. Cash only, no ID."),
    ("F14", "07-04", "Fantasyland - Carousel",    "Plush toy, pink, bear, new condition w/ tags."),
    ("F15", "07-05", "Frontierland",              "Sunglasses, black plastic frames, no case."),
    ("F16", "07-05", "Adventureland",             "Silver bracelet, chain link."),
    ("F17", "07-04", "Main St - Emporium",        "Headband, pink sequin bow, bunny ears (not Minnie). No name."),
    ("F18", "07-04", "Adventureland - Pirates",   "Backpack, navy blue, Patagonia logo, zippers OK, empty, no keys."),
    ("F19", "07-05", "Tomorrowland - Speedway",   "Water bottle, red Hydro Flask, no stickers, dented."),
    ("F20", "07-04", "Fantasyland - Peter Pan",   "Stuffed Stitch toy, new with tags, gift shop bag."),
    ("F21", "07-05", "Liberty Sq",                "Baby blanket, yellow fleece, star print, machine made."),
    ("F22", "07-05", "Frontierland - Big Thunder","Apple AirPods case, white, orange sticker."),
    ("F23", "07-05", "Main St - Casey's",         "Coach wallet, brown leather, empty, no ID or cards."),
], columns=["found_id", "date", "location", "description"])

GOLD = {"L01": "F03", "L02": "F05", "L03": "F09", "L04": "F01", "L05": "F02", "L06": "F04",
        "L07": "F10", "L08": "F06", "L09": "F07", "L10": "F08", "L11": "F11", "L12": None}
display(LOST); display(FOUND)

### Step 9.1: Baseline (retrieval only)
Embed the raw text and match each lost report to its nearest found item. This is the number to beat.

In [ ]:
def as_text(row):
    return f"{row.description} Location: {row.location}. Date: {row.date}."

lost_txt, found_txt = LOST.apply(as_text, axis=1).tolist(), FOUND.apply(as_text, axis=1).tolist()
S = cosine_similarity(embed(lost_txt, fit_corpus=lost_txt + found_txt),
                      embed(found_txt, fit_corpus=lost_txt + found_txt))

def eval_matches(pred, label):
    correct = sum(pred.get(l) == g for l, g in GOLD.items())
    false_match = sum(pred.get(l) is not None and g is None for l, g in GOLD.items())
    print(f"{label}: {correct}/{len(GOLD)} correct | false matches on NO_MATCH items: {false_match}")

def recall_at_k(S, k):
    hits = [GOLD[l] in FOUND.found_id.iloc[np.argsort(-S[i])[:k]].tolist()
            for i, l in enumerate(LOST.lost_id) if GOLD[l]]
    return np.mean(hits)

baseline = {l: FOUND.found_id.iloc[S[i].argmax()] for i, l in enumerate(LOST.lost_id)}
eval_matches(baseline, f"Baseline ({EMBED_BACKEND} nearest neighbor)")
for k in [1, 3, 5]:
    print(f"recall@{k} = {recall_at_k(S, k):.2f}")

### Step 9.2: ✏️ TODO: Normalize with an LLM
Write a prompt that converts **any** report (guest or staff) into the **same JSON schema**. Normalizing vocabulary ("Stitch" → "plush toy, blue alien") before embedding usually improves retrieval a lot. Re-run retrieval on the normalized text and compare recall@k with the baseline.

In [ ]:
NORMALIZE_PROMPT = """You are a lost-and-found clerk at a theme park. Convert the report below into a
standard record so that a guest's report and a staff log describing the SAME item look alike.

Return ONLY JSON with these keys:
{{"category": generic item type (e.g. "mobile phone", "plush toy", "eyeglasses", "wallet"),
  "color": main color(s) or null,
  "brand_or_character": brand or character name or null,
  "generic_description": the item described WITHOUT brand/character names (e.g. Stitch -> "blue alien character plush"),
  "distinguishing_features": list of specific marks, damage, engravings, writing, or contents,
  "condition": "new" | "used" | "worn" | "damaged" | null,
  "location": park area}}

Rules:
- Never invent details that are not in the report; use null when unknown.
- Do NOT copy personal names, license numbers, or other personal identifiers; write e.g. "name written inside" instead.

Report: {report}"""

def normalize(text, tag):
    p = parse_json(chat(NORMALIZE_PROMPT.format(report=text), tag=tag, show=False))
    return json.dumps(p) if isinstance(p, dict) else text   # fall back to raw text if parsing fails

lost_norm  = [normalize(t, "cap-norm-lost") for t in lost_txt]
found_norm = [normalize(t, "cap-norm-found") for t in found_txt]
S_norm = cosine_similarity(embed(lost_norm, fit_corpus=lost_norm + found_norm),
                           embed(found_norm, fit_corpus=lost_norm + found_norm))

norm_baseline = {l: FOUND.found_id.iloc[S_norm[i].argmax()] for i, l in enumerate(LOST.lost_id)}
eval_matches(norm_baseline, "Normalized + nearest neighbor")
for k in [1, 3, 5]:
    print(f"recall@{k}: raw = {recall_at_k(S, k):.2f} | normalized = {recall_at_k(S_norm, k):.2f}")
print("\nExample:", lost_norm[3])

### Step 9.3: ✏️ TODO: Rerank with an LLM
For each lost report, give the model the top-k candidates and ask for JSON `{"match": "<found_id or NO_MATCH>", "confidence": 0-1, "reason": "..."}`. Design the prompt so it **rejects** candidates that conflict on hard attributes (color, brand, item type) and uses `NO_MATCH` when nothing fits. That is the hallucination guardrail from Part 4, applied to a business task.

In [ ]:
RERANK_PROMPT = """You are an expert lost-and-found matcher at a theme park. Decide which candidate
found item, if any, is the SAME physical object described in the lost report.

Check every candidate against these rules:
- HARD attributes must agree: item type, color, brand/character, and any unique marks
  (engravings, writing, damage, contents, condition). A contradiction on ANY hard attribute rules it out.
- A detail that is simply missing from a terse staff log is NOT a contradiction.
- Dates and locations should be plausible (items get moved), but they are weaker evidence than the item itself.
- If no candidate survives, answer NO_MATCH. A wrong match is worse than no match.

Return ONLY JSON: {{"match": "<found_id or NO_MATCH>", "confidence": <0-1>,
"reason": "<one sentence naming the deciding evidence>"}}

Lost report:
{lost}

Candidate found items:
{candidates}
"""

def rerank(S, k=3, temperature=0.0):
    preds, reasons = {}, {}
    for i, row in LOST.iterrows():
        idx = np.argsort(-S[i])[:k]
        cands = "\n".join(f"- {FOUND.found_id.iloc[j]}: {found_txt[j]}" for j in idx)
        out = parse_json(chat(RERANK_PROMPT.format(lost=lost_txt[i], candidates=cands),
                              temperature=temperature, tag="cap-rerank", show=False)) or {}
        m = out.get("match")
        preds[row.lost_id] = None if m in (None, "NO_MATCH") else m
        reasons[row.lost_id] = out.get("reason")
    return preds, reasons

preds, reasons = rerank(S_norm, k=3)
eval_matches(preds, "Normalize + retrieve + LLM rerank")
pd.DataFrame({"pred": preds, "gold": GOLD, "reason": reasons})

> 🧑‍🏫 **Grading notes (Part 9.2–9.3):** The decoys are designed to defeat keyword matching, and each one tests a specific rule:
>
> | Lost | Correct | Decoy | Why the decoy is wrong |
> |---|---|---|---|
> | L01 Minnie ears, name inside | F03 | F17 | bunny ears, no name |
> | L03 Patagonia backpack, broken zipper, keys | F09 | F18 | zippers OK, no keys |
> | L04 *worn* Stitch, ear damaged | F01 ("blue alien", worn) | F20, F14 | new with tags |
> | L06 Coach wallet with CO license | F04 | F23, F13 | empty / black nylon |
> | L07 *hand-knit* gray star blanket | F10 | F21 | yellow fleece, machine made |
> | L08 Hydro Flask *with stickers* | F06 | F19 | no stickers |
> | L12 silver Apple Watch, orange band | **NO_MATCH** | F22, F16 | AirPods case / bracelet |
>
> The baseline typically falls for the brand-name decoys (F20 "Stitch", F23 "Coach", F18 "Patagonia") because they share rare tokens. Normalization fixes L04, where the guest says "Stitch" and staff wrote "blue alien". The rerank fixes most of the rest. Expect **≥ 10/12 with 0 false matches** from a capable model at k=3. If recall@3 < 1 for a student, their rerank can't recover that item. Check whether they reasoned about the k vs accuracy trade-off. Award the PII rule in the normalization prompt explicitly.

### Step 9.4: Business evaluation
✏️ **TODO:** Fill in the table and answer the questions in a short memo (≤ 200 words) addressed to the Lost and Found department.

| Approach | Correct / 12 | False matches | LLM calls | Notes |
|---|---|---|---|---|
| Baseline embeddings |  |  | 0 |  |
| + LLM normalization |  |  |  |  |
| + LLM rerank |  |  |  |  |

1. Does this meet the **50% time reduction** goal? What workflow would you propose: auto-match, or *suggest top-3 to a staff member*?
2. What happens at scale (thousands of reports a day)? Where do cost and latency go?
3. **PII:** lost reports contain names, license details, and phone photos. What must you do before sending this data to a hosted model? (See the lecture's *"Yes, but… confidential PII data"* slide.)
4. What would you need to persist (the vector index) and refresh as new items arrive? This leads into next week's **LangChain + FAISS / RAG** material.

**Model answer (fill the counts from your calibration run):**

| Approach | Correct / 12 | False matches | LLM calls | Notes |
|---|---|---|---|---|
| Baseline embeddings | ~5 (TF-IDF) / higher with real embeddings | 1 (L12 always gets a match) | 0 | Fooled by shared brand/character tokens |
| + LLM normalization | improves, esp. L04 | 1 | 35 (12 + 23) | Still no way to say "no match" |
| + LLM rerank | ≥ 10 | 0 | 35 + 12 = 47 | Hard-attribute rules plus NO_MATCH |

> **To:** Lost and Found Matching Dept. · **Re:** GenAI matching pilot
>
> On a 12-report test set, our pipeline (normalize → retrieve top-3 → LLM verify) matched ≥10 items correctly and made **no false matches**, versus 5/12 for keyword search. We recommend a **suggest, don't decide** workflow: staff see the top-3 candidates with the model's reason and confirm with one click. Low-confidence or NO_MATCH cases go to manual search. This should cut per-report handling time well beyond the 50% target, while a human remains accountable for every return of high-value items (rings, IDs, phones). **At scale**, embeddings are computed once per report and stored in a vector index (FAISS or a managed vector DB) that is updated as items arrive. LLM calls grow linearly (~4 per lost report), so cost is cents per report, and a normalization step can run in batch overnight. **Privacy:** reports contain names, license numbers, and phone photos. Redact PII before any call to a hosted model, use an enterprise endpoint with no training on our data and an appropriate data-processing agreement, and set retention limits. Before rollout, validate on a few hundred historical matched cases and monitor weekly precision.

> 🧑‍🏫 **Grading notes:** Five memo elements, 1 point each: (1) HITL workflow, (2) the 50% claim tied to evidence, (3) scale/cost, (4) PII handling, (5) index persistence/refresh. Deduct if the memo claims full automation with no human confirmation for high-value items.

---
## Part 10: Prompt as an asset (Prompt Lab, Exercise 6)

Turn your best capstone prompt into a **reusable, documented template**. Include: role, inputs (as variables), output schema, safety rules, and an example. Save it as a file so it can be versioned, like code.

In [ ]:
_n_correct = sum(preds.get(l) == g for l, g in GOLD.items()) if "preds" in globals() else None
_n_false = sum(preds.get(l) is not None and g is None for l, g in GOLD.items()) if "preds" in globals() else None

TEMPLATE = {
    "name": "lost_found_matcher",
    "version": "1.0",
    "model": CHAT_MODEL,
    "parameters": {"temperature": 0.0, "max_tokens": 300},
    "role": "Expert lost-and-found matcher at a theme park; verifies whether a found item is the same "
            "physical object as a guest's lost report.",
    "inputs": ["lost_report", "candidates"],
    "output_schema": {"match": "found_id | NO_MATCH", "confidence": "float 0-1", "reason": "string"},
    "safety_rules": [
        "Never output personal identifiers (names, license or card numbers) in the reason field.",
        "Prefer NO_MATCH over a guess; a wrong match is worse than none.",
        "Inputs must be PII-redacted before calling a hosted model.",
        "Matches of high-value items (jewelry, IDs, phones, wallets) require staff confirmation.",
    ],
    "prompt": RERANK_PROMPT,
    "eval": {"dataset": "week8 synthetic lost/found (12 lost, 23 found)", "retrieval": f"{EMBED_BACKEND} top-3",
             "top1_correct": _n_correct, "false_matches": _n_false},
}
with open("prompt_template_lost_found.json", "w") as f:
    json.dump(TEMPLATE, f, indent=2, default=str)
print("Saved prompt_template_lost_found.json")

## Wrap-up: export your log and reflect

In [ ]:
log = pd.DataFrame(PROMPT_LOG)
log.to_csv("prompt_log.csv", index=False)
print(f"{len(log)} calls logged → prompt_log.csv")
log.groupby(log.tag.str.extract(r"^(ex\d|cap)", expand=False)).agg(
    calls=("tag", "size"), mean_latency_s=("latency_s", "mean"))

💭 **Final reflection (≤ 250 words):**
1. Which single prompt change had the largest **measured** effect today?
2. Where did prompting stop being enough, so that you needed embeddings, multiple calls, or a human in the loop?
3. Name one way your capstone system could cause harm to a guest, and how you'd mitigate it.

**Model answer (example):**
1. The largest measured effect was adding an **explicit abstention rule** (Part 4: 2/5 → 5/5) and, in the capstone, the **rerank with NO_MATCH** (5/12 → ≥10/12 with zero false matches). Both are about letting the model *decline*, not about making it smarter.
2. Prompting stopped being enough once the task needed **search over many items** (embeddings for the shortlist), **verifiable multi-step reasoning** (multi-call or code for arithmetic), and **accountability** (a human confirms high-value returns). The prompt is one component of a system.
3. **Harm:** a false match could hand a guest's wallet or ID to the wrong person, a privacy and theft risk. **Mitigation:** require proof-of-ownership questions generated from details withheld from the claimant (e.g. the ring's engraving), staff confirmation for high-value categories, and logging for audit.

> 🧑‍🏫 **Grading notes:** Full credit requires citing **their own numbers**. Answers that read like generic AI commentary earn at most 1/3.

---
### ✅ Deliverables checklist
- [ ] This notebook, fully run, with all ✏️ TODOs completed and 💭 reflections answered
- [ ] `prompt_log.csv` (your experiment record)
- [ ] `prompt_template_lost_found.json` (your saved template)
- [ ] Capstone results table + memo (Part 9.4)

**Optional extension:** port your template to **IBM watsonx Prompt Lab** (or Azure AI Foundry playground), run it against a different base model, and compare the results with a before/after screenshot.

**Next up:** Prompting → **RAG** → Agents → Deployment